# Notebook 26. Timing-Group Manual Verification Atlas

This notebook is the manual-verification follow-on to `Notebook 25`.

Its review unit is a **catalog timing group**, not a single event. For one selected timing group, it shows:

- timing-group header and timing summary
- event membership table with all events inside the group
- first-look event-peak plots:
  - `925 hPa` signed divergence
  - `850 hPa q × (-omega)` moist-ascent proxy
  - saved quicklook / OLR / satellite panels when available
- broader “forecast-funnel” synoptic context:
  - `300 hPa` isotachs with `300 hPa` geopotential-height contours
  - `500 hPa` geopotential-height contours
  - surface mean sea-level pressure with `850 hPa` temperature-gradient shading
- continuous timing-group evolution from `Notebook 25`
- a timing-group manual-review scaffold row for later note-taking
- a cross-section section reserved for the next implementation layer

## Plotting standard for this notebook

To keep the figures readable and consistent with the earlier professor feedback:

- colorbars use **display-only caps** rather than per-event autoscaling
- signed fields use **fixed symmetric limits around 0**
- one-sided fields use **fixed upper caps**
- colorbars are placed **below the panels**, not over the map axes
- legends are placed **outside the data region**
- every plotted quantity is labeled with its meteorological name and units

This notebook is designed as the timing-group manual-review atlas that comes **before** the later cross-section refinement pass.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")


def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )


def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
print("Working directory:", os.getcwd())
print("Runtime repo branch:", active_branch)


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, OBJECTIVE_SUBTYPE_DOMAIN, BoundingBox
from jpcz_catalog.diagnostics import (
    load_snapshot,
    compute_geopotential_height_field,
    compute_temperature_gradient_magnitude,
    compute_wind_speed_field,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.objective_regimes import assign_objective_labels_from_thresholds
from jpcz_catalog.satellite import default_modis_layers_for_date

OBJECTIVE_EXPORT_DIR = Path("outputs/verification/objective_coastal_box_regimes")
TIMING_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_and_impact")
EOF_EXPORT_DIR = Path("outputs/verification/objective_subtype_eof_analysis")
REVIEW_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_group_manual_review")
REVIEW_PLOT_DIR = Path("outputs/verification/objective_regime_timing_group_manual_review_plots")
REVIEW_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_PLOT_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE_EVENT_METRICS_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_event_metrics.csv"
LOW_LEVEL_STACK_PATH = EOF_EXPORT_DIR / "cleaned_low_level_eof_event_fields.nc"
TIMING_GROUP_REVIEW_SCAFFOLD_PATH = REVIEW_EXPORT_DIR / "objective_timing_group_manual_review_scaffold.csv"

QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")

CONTINUOUS_RUN_TAG = "fullrun_allprofiles_allspells"
CONTINUOUS_SPELL_GAP_HOURS = 36
CONTINUOUS_PADDING_HOURS = 48
CONTINUOUS_TIME_STEP_HOURS = 6
REVIEW_THRESHOLD_PROFILE = "coastal_permissive"
REVIEW_TIMING_GROUP_ID = None
REVIEW_PREFERRED_PATHS = [
    "offshore_to_coastal",
    "coastal_to_offshore",
    "mixed_or_oscillating",
    "offshore_only",
    "coastal_only",
    "weak_only",
]
SYNOPTIC_MAX_TIMES = 3
ERA5_TIME_CHUNK = 24
SAVE_PLOTS = False

COASTAL_STRIP_VERTICES = (
    (133.05, 35.55),
    (136.05, 35.55),
    (139.55, 39.00),
    (139.55, 42.55),
)
UPPER_CONTEXT_DOMAIN = BoundingBox(lon_min=110.0, lon_max=170.0, lat_min=25.0, lat_max=60.0)
PRESSURE_CONTEXT_DOMAIN = BoundingBox(lon_min=80.0, lon_max=230.0, lat_min=20.0, lat_max=80.0)

CONTINUOUS_CANDIDATE_THRESHOLDS_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_candidate_thresholds_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SPELL_EVENT_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_events_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SPELL_WINDOW_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_windows_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_LABELED_TIMESERIES_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_labeled_spell_timeseries_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_PROFILE_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_profile_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"

MOISTURE_FIELD = "vertical_moisture_flux_proxy_850_peak"
DIVERGENCE_FIELD = "divergence_925_peak"

EVENT_FIELD_SPECS = [
    {
        "field_name": MOISTURE_FIELD,
        "panel_title": "850 hPa q × (-omega) moist-ascent proxy",
        "colorbar_label": "850 hPa q × (-omega) moist-ascent proxy [1e-3 Pa s^-1]",
        "cmap": "BrBG",
        "cap": 0.6,
        "mode": "symmetric",
        "extend": "both",
    },
    {
        "field_name": DIVERGENCE_FIELD,
        "panel_title": "925 hPa signed divergence",
        "colorbar_label": "925 hPa signed divergence [1e-5 s^-1]",
        "cmap": "RdBu_r",
        "cap": 2.0,
        "mode": "symmetric",
        "extend": "both",
    },
]

STATE_COLOR_MAP = {
    "offshore_jpcz_core": "#4c78a8",
    "coastal_interaction": "#f58518",
    "mixed_transition": "#9c755f",
    "weak_or_unclear": "#bab0ac",
}


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive cache:", drive_path)
    return True


def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()


def read_csv_with_dates(path: Path, candidate_date_columns: list[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)
    parse_dates = [column for column in candidate_date_columns if column in header.columns]
    return pd.read_csv(path, parse_dates=parse_dates)


def quicklook_filename(event_index: int, event_peak: pd.Timestamp) -> str:
    return f"{pd.Timestamp(event_peak).strftime('%Y%m%dT%H%M')}_idx{int(event_index):03d}.png"


def satellite_filename(event_index: int, event_peak: pd.Timestamp) -> str | None:
    layers = default_modis_layers_for_date(pd.Timestamp(event_peak).normalize())
    if not layers:
        return None
    layer_slug = (
        layers[0].replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    timestamp = pd.Timestamp(event_peak).strftime("%Y%m%dT%H%M")
    return f"{timestamp}_idx{int(event_index):03d}_{layer_slug}.jpg"


def first_existing_path(local_dir: Path, filename: str | None) -> Path | None:
    if filename is None:
        return None
    local_path = local_dir / filename
    if local_path.exists():
        return local_path
    if PERSIST_OUTPUTS_TO_DRIVE:
        drive_path = Path(DRIVE_OUTPUT_DIR) / local_dir.name / filename
        if drive_path.exists():
            return drive_path
    return None


def configure_map_axis(ax, domain: BoundingBox):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="50m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.35)
    ax.add_feature(cfeature.LAND, facecolor="#efefef", alpha=0.45)
    gl = ax.gridlines(draw_labels=True, linestyle=":", linewidth=0.35, alpha=0.45)
    gl.top_labels = False
    gl.right_labels = False
    return ax


def add_region_outlines(ax, *, include_polygon: bool = True, include_coastal: bool = True):
    handles = []
    labels = []
    if include_polygon:
        polygon_vertices = np.array(JPCZ_POLYGON_VERTICES)
        line, = ax.plot(
            np.r_[polygon_vertices[:, 0], polygon_vertices[0, 0]],
            np.r_[polygon_vertices[:, 1], polygon_vertices[0, 1]],
            color="#1f77b4",
            linewidth=1.9,
            transform=ccrs.PlateCarree(),
        )
        handles.append(line)
        labels.append("JPCZ polygon")
    if include_coastal:
        coastal_vertices = np.array(COASTAL_STRIP_VERTICES)
        line, = ax.plot(
            np.r_[coastal_vertices[:, 0], coastal_vertices[0, 0]],
            np.r_[coastal_vertices[:, 1], coastal_vertices[0, 1]],
            color="#d95f02",
            linewidth=1.9,
            transform=ccrs.PlateCarree(),
        )
        handles.append(line)
        labels.append("Coastal wedge")
    return handles, labels


def unique_time_specs(items: list[tuple[str, pd.Timestamp | str | None]], *, limit: int | None = None):
    seen = set()
    ordered = []
    for label, value in items:
        if value is None or pd.isna(value):
            continue
        timestamp = pd.Timestamp(value)
        if timestamp in seen:
            continue
        seen.add(timestamp)
        ordered.append((label, timestamp))
        if limit is not None and len(ordered) >= limit:
            break
    return ordered


def rounded_contour_levels(field: xr.DataArray, *, step: float) -> np.ndarray:
    finite = np.asarray(field.values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.array([0.0])
    lower = np.floor(finite.min() / step) * step
    upper = np.ceil(finite.max() / step) * step
    if lower == upper:
        upper = lower + step
    return np.arange(lower, upper + 0.5 * step, step)


print("Notebook 26 configuration")
print(f"- Continuous run tag: {CONTINUOUS_RUN_TAG}")
print(f"- Review threshold profile: {REVIEW_THRESHOLD_PROFILE}")
print(f"- Review timing-group override: {REVIEW_TIMING_GROUP_ID}")
print(f"- Synoptic max representative times: {SYNOPTIC_MAX_TIMES}")
print("- Plotting rule: fixed display-only caps with horizontal colorbars and outside legends")


In [ ]:
paths_to_restore = [
    OBJECTIVE_EVENT_METRICS_PATH,
    LOW_LEVEL_STACK_PATH,
    CONTINUOUS_CANDIDATE_THRESHOLDS_PATH,
    CONTINUOUS_SPELL_EVENT_PATH,
    CONTINUOUS_SPELL_WINDOW_PATH,
    CONTINUOUS_LABELED_TIMESERIES_PATH,
    CONTINUOUS_SUMMARY_PATH,
    CONTINUOUS_PROFILE_SUMMARY_PATH,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

required_paths = {
    "Notebook 22 objective event metrics": OBJECTIVE_EVENT_METRICS_PATH,
    "Notebook 17 low-level event stack": LOW_LEVEL_STACK_PATH,
    "Notebook 25 candidate thresholds": CONTINUOUS_CANDIDATE_THRESHOLDS_PATH,
    "Notebook 25 timing-group membership": CONTINUOUS_SPELL_EVENT_PATH,
    "Notebook 25 timing-group windows": CONTINUOUS_SPELL_WINDOW_PATH,
    "Notebook 25 labeled timing-group time series": CONTINUOUS_LABELED_TIMESERIES_PATH,
    "Notebook 25 timing-group summary": CONTINUOUS_SUMMARY_PATH,
    "Notebook 25 profile summary": CONTINUOUS_PROFILE_SUMMARY_PATH,
}
missing_paths = [f"{label}: {path}" for label, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise RuntimeError("Missing required Notebook 26 inputs:\n- " + "\n- ".join(missing_paths))

objective_df = read_csv_with_dates(
    OBJECTIVE_EVENT_METRICS_PATH,
    [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
    ],
)
if "event_index" not in objective_df.columns:
    objective_df["event_index"] = objective_df.index.astype(int)

candidate_thresholds_df = pd.read_csv(CONTINUOUS_CANDIDATE_THRESHOLDS_PATH)
catalog_spell_events_df = read_csv_with_dates(
    CONTINUOUS_SPELL_EVENT_PATH,
    [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
    ],
)
catalog_spell_window_df = read_csv_with_dates(
    CONTINUOUS_SPELL_WINDOW_PATH,
    [
        "spell_start",
        "spell_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "analysis_start",
        "analysis_end",
    ],
)
continuous_labeled_timeseries_df = read_csv_with_dates(
    CONTINUOUS_LABELED_TIMESERIES_PATH,
    [
        "time",
        "spell_start",
        "spell_end",
        "analysis_start",
        "analysis_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "first_offshore_support_time",
        "first_coastal_support_time",
    ],
)
continuous_summary_df = read_csv_with_dates(
    CONTINUOUS_SUMMARY_PATH,
    [
        "spell_start",
        "spell_end",
        "analysis_start",
        "analysis_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "first_offshore_support_time",
        "first_coastal_support_time",
    ],
)
continuous_profile_summary_df = pd.read_csv(CONTINUOUS_PROFILE_SUMMARY_PATH)
low_level_stack_ds = load_cached_dataset(LOW_LEVEL_STACK_PATH)
if low_level_stack_ds is None:
    raise RuntimeError("Unable to restore the low-level event stack from Notebook 17.")

objective_df = add_japan_local_time_columns(objective_df)

print("Notebook 26 inputs loaded")
print(f"- Objective events available: {len(objective_df)}")
print(f"- Timing groups available: {catalog_spell_window_df['catalog_spell_id'].nunique()}")
print(f"- Labeled continuous timing-group rows: {len(continuous_labeled_timeseries_df)}")
print(f"- Threshold profiles available: {candidate_thresholds_df['threshold_profile'].nunique()}")
print(f"- Low-level event stack event count: {low_level_stack_ds.sizes.get('event_index', 'unknown')}")


In [ ]:
required_globals = [
    "objective_df",
    "candidate_thresholds_df",
    "catalog_spell_events_df",
    "catalog_spell_window_df",
    "continuous_labeled_timeseries_df",
    "continuous_summary_df",
    "continuous_profile_summary_df",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 setup cells first. Missing globals: {missing_globals}")

selected_threshold_row = candidate_thresholds_df.loc[
    candidate_thresholds_df["threshold_profile"] == REVIEW_THRESHOLD_PROFILE
]
if selected_threshold_row.empty:
    raise RuntimeError(f"REVIEW_THRESHOLD_PROFILE={REVIEW_THRESHOLD_PROFILE!r} was not found in the Notebook 25 threshold table.")
selected_threshold_row = selected_threshold_row.iloc[0]

profile_subset_df = continuous_summary_df.loc[
    continuous_summary_df["threshold_profile"] == REVIEW_THRESHOLD_PROFILE
].copy()
if profile_subset_df.empty:
    raise RuntimeError(f"No continuous timing-group summaries were found for REVIEW_THRESHOLD_PROFILE={REVIEW_THRESHOLD_PROFILE!r}.")

if REVIEW_TIMING_GROUP_ID is not None:
    selected_group_row = profile_subset_df.loc[
        profile_subset_df["catalog_spell_id"] == REVIEW_TIMING_GROUP_ID
    ]
    if selected_group_row.empty:
        raise RuntimeError(
            f"REVIEW_TIMING_GROUP_ID={REVIEW_TIMING_GROUP_ID!r} was not found for profile {REVIEW_THRESHOLD_PROFILE!r}."
        )
    selected_group_row = selected_group_row.iloc[0]
else:
    selected_group_row = None
    for preferred_path in REVIEW_PREFERRED_PATHS:
        subset = profile_subset_df.loc[
            profile_subset_df["continuous_spell_regime_path"] == preferred_path
        ]
        if not subset.empty:
            selected_group_row = subset.sort_values(
                ["event_count", "analysis_window_hours"],
                ascending=[False, False],
            ).iloc[0]
            break
    if selected_group_row is None:
        selected_group_row = profile_subset_df.sort_values(
            ["event_count", "analysis_window_hours"],
            ascending=[False, False],
        ).iloc[0]

selected_group_id = str(selected_group_row["catalog_spell_id"])
selected_group_events_df = catalog_spell_events_df.loc[
    catalog_spell_events_df["catalog_spell_id"].astype(str) == selected_group_id
].sort_values("event_peak").reset_index(drop=True)
selected_group_window_row = catalog_spell_window_df.loc[
    catalog_spell_window_df["catalog_spell_id"].astype(str) == selected_group_id
].iloc[0]
selected_group_timeseries_df = continuous_labeled_timeseries_df.loc[
    (continuous_labeled_timeseries_df["threshold_profile"] == REVIEW_THRESHOLD_PROFILE)
    & (continuous_labeled_timeseries_df["catalog_spell_id"].astype(str) == selected_group_id)
].sort_values("time").reset_index(drop=True)
selected_group_all_profiles_df = continuous_summary_df.loc[
    continuous_summary_df["catalog_spell_id"].astype(str) == selected_group_id
].copy()
selected_group_all_profiles_df["threshold_profile"] = pd.Categorical(
    selected_group_all_profiles_df["threshold_profile"],
    categories=candidate_thresholds_df["threshold_profile"].tolist(),
    ordered=True,
)
selected_group_all_profiles_df = selected_group_all_profiles_df.sort_values("threshold_profile").reset_index(drop=True)

objective_metric_columns = [
    "event_index",
    "objective_regime_label",
    "polygon_qflux_850_mean",
    "coastal_qflux_850_mean",
    "polygon_div_925_mean",
    "coastal_div_925_mean",
]
selected_profile_event_df = (
    selected_group_events_df.drop(columns=["objective_regime_label"], errors="ignore")
    .merge(
        objective_df[[column for column in objective_metric_columns if column in objective_df.columns]],
        on="event_index",
        how="left",
    )
    .rename(columns={"objective_regime_label": "default_threshold_label"})
)
selected_profile_event_df = assign_objective_labels_from_thresholds(
    selected_profile_event_df.copy(),
    polygon_qflux_min=float(selected_threshold_row["polygon_qflux_min"]),
    polygon_div_max=float(selected_threshold_row["polygon_div_max"]),
    coastal_qflux_split=float(selected_threshold_row["coastal_qflux_split"]),
    coastal_div_max=float(selected_threshold_row["coastal_div_max"]),
).rename(columns={"objective_regime_label": "selected_profile_event_label"})

representative_time_specs = unique_time_specs(
    [
        ("first event peak", selected_group_row.get("first_event_peak")),
        ("first offshore-support onset", selected_group_row.get("first_offshore_support_time")),
        ("first coastal-support onset", selected_group_row.get("first_coastal_support_time")),
        ("last event peak", selected_group_row.get("last_event_peak")),
    ],
    limit=SYNOPTIC_MAX_TIMES,
)
representative_time_table_df = pd.DataFrame(
    {
        "time_role": [label for label, _ in representative_time_specs],
        "time_utc": [timestamp for _, timestamp in representative_time_specs],
    }
)
if not representative_time_table_df.empty:
    representative_time_table_df["time_jst"] = representative_time_table_df["time_utc"] + pd.Timedelta(hours=9)

threshold_display_df = pd.DataFrame(
    {
        "threshold_profile": [REVIEW_THRESHOLD_PROFILE],
        "polygon_qflux_min": [float(selected_threshold_row["polygon_qflux_min"])],
        "coastal_qflux_split": [float(selected_threshold_row["coastal_qflux_split"])],
        "polygon_div_max": [float(selected_threshold_row["polygon_div_max"])],
        "coastal_div_max": [float(selected_threshold_row["coastal_div_max"])],
    }
)

selected_group_header_df = pd.DataFrame(
    [
        {
            "catalog_spell_id": selected_group_id,
            "threshold_profile": REVIEW_THRESHOLD_PROFILE,
            "continuous_spell_regime_path": selected_group_row["continuous_spell_regime_path"],
            "continuous_clear_sequence": selected_group_row["continuous_clear_sequence"],
            "event_count": int(selected_group_row["event_count"]),
            "spell_start": selected_group_row["spell_start"],
            "spell_end": selected_group_row["spell_end"],
            "analysis_start": selected_group_row["analysis_start"],
            "analysis_end": selected_group_row["analysis_end"],
            "first_event_peak_jst": selected_group_row.get("first_event_peak_jst", pd.NaT),
            "last_event_peak_jst": selected_group_row.get("last_event_peak_jst", pd.NaT),
            "first_offshore_support_time": selected_group_row.get("first_offshore_support_time", pd.NaT),
            "first_coastal_support_time": selected_group_row.get("first_coastal_support_time", pd.NaT),
            "offshore_to_coastal_lag_hours": selected_group_row.get("offshore_to_coastal_lag_hours", np.nan),
            "coastal_to_offshore_lag_hours": selected_group_row.get("coastal_to_offshore_lag_hours", np.nan),
            "spell_duration_hours": float(selected_group_row.get("spell_duration_hours", np.nan)),
            "analysis_window_hours": float(selected_group_row.get("analysis_window_hours", np.nan)),
        }
    ]
)

selected_event_display_df = selected_profile_event_df[[
    column for column in [
        "event_index",
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
        "default_threshold_label",
        "selected_profile_event_label",
        "offshore_rule_pass",
        "coastal_rule_pass",
        "polygon_qflux_850_mean",
        "coastal_qflux_850_mean",
        "polygon_div_925_mean",
        "coastal_div_925_mean",
        "gap_from_previous_event_hours",
    ]
    if column in selected_profile_event_df.columns
]].copy()

print("Notebook 26 selected timing-group review target")
print(f"- Review threshold profile: {REVIEW_THRESHOLD_PROFILE}")
print(f"- Selected catalog timing group: {selected_group_id}")
print()
print("Timing-group header")
display(selected_group_header_df)
print("Threshold values active in this review")
display(threshold_display_df)
print("How this timing group behaves across all three threshold profiles")
display(selected_group_all_profiles_df[[
    "threshold_profile",
    "continuous_spell_regime_path",
    "continuous_clear_sequence",
    "event_count",
    "offshore_to_coastal_lag_hours",
    "coastal_to_offshore_lag_hours",
]])
print("Event membership table for the selected timing group")
display(selected_event_display_df)
if not representative_time_table_df.empty:
    print("Representative synoptic times chosen for the forecast-funnel maps")
    display(representative_time_table_df)
else:
    print("No representative synoptic times were available for this timing group.")


In [ ]:
required_globals = [
    "selected_group_id",
    "selected_profile_event_df",
    "low_level_stack_ds",
    "EVENT_FIELD_SPECS",
    "SAVE_PLOTS",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 selection cells first. Missing globals: {missing_globals}")

for event_row in selected_profile_event_df.itertuples(index=False):
    event_index = int(event_row.event_index)
    event_peak = pd.Timestamp(event_row.event_peak)
    event_peak_jst = pd.Timestamp(event_row.event_peak_jst) if pd.notna(event_row.event_peak_jst) else pd.NaT
    event_ds = low_level_stack_ds.sel(event_index=event_index)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13.8, 5.9),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    fig.subplots_adjust(top=0.82, bottom=0.20, left=0.05, right=0.98, wspace=0.08)

    legend_handles = None
    legend_labels = None
    for ax, spec in zip(axes, EVENT_FIELD_SPECS):
        field = event_ds[spec["field_name"]]
        configure_map_axis(ax, OBJECTIVE_SUBTYPE_DOMAIN)
        if spec["mode"] == "symmetric":
            norm = mcolors.TwoSlopeNorm(vmin=-spec["cap"], vcenter=0.0, vmax=spec["cap"])
            mesh = ax.pcolormesh(
                field.longitude,
                field.latitude,
                field,
                cmap=spec["cmap"],
                norm=norm,
                shading="auto",
                transform=ccrs.PlateCarree(),
            )
        else:
            mesh = ax.pcolormesh(
                field.longitude,
                field.latitude,
                field,
                cmap=spec["cmap"],
                vmin=0.0,
                vmax=spec["cap"],
                shading="auto",
                transform=ccrs.PlateCarree(),
            )
        legend_handles, legend_labels = add_region_outlines(ax)
        ax.set_title(spec["panel_title"], fontsize=11)

        bbox = ax.get_position()
        cbar_ax = fig.add_axes([bbox.x0, 0.085, bbox.width, 0.025])
        cbar = fig.colorbar(mesh, cax=cbar_ax, orientation="horizontal", extend=spec["extend"])
        cbar.set_label(spec["colorbar_label"] + " (display-only cap)")

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.92),
            ncol=2,
            frameon=False,
        )
    fig.suptitle(
        f"Timing group {selected_group_id} | event {event_index:03d} | peak {event_peak:%Y-%m-%d %H:%M UTC}"
        + (f" / {event_peak_jst:%Y-%m-%d %H:%M JST}" if pd.notna(event_peak_jst) else "")
        + f"\ndefault label: {event_row.default_threshold_label} | {REVIEW_THRESHOLD_PROFILE}: {event_row.selected_profile_event_label}",
        fontsize=13,
        y=0.98,
    )

    if SAVE_PLOTS:
        output_path = REVIEW_PLOT_DIR / f"{selected_group_id}_event_{event_index:03d}_firstlook_maps.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(output_path, verbose=False)
    plt.show()


In [ ]:
required_globals = [
    "selected_profile_event_df",
    "selected_group_id",
    "QUICKLOOK_DIR",
    "OLR_DIR",
    "SATELLITE_DIR",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 selection cells first. Missing globals: {missing_globals}")

for event_row in selected_profile_event_df.itertuples(index=False):
    event_index = int(event_row.event_index)
    event_peak = pd.Timestamp(event_row.event_peak)
    event_peak_jst = pd.Timestamp(event_row.event_peak_jst) if pd.notna(event_row.event_peak_jst) else pd.NaT

    header_md = f"""## Timing group `{selected_group_id}` | event `{event_index:03d}`

Peak: `{event_peak:%Y-%m-%d %H:%M UTC}`"""
    if pd.notna(event_peak_jst):
        header_md += f" / `{event_peak_jst:%Y-%m-%d %H:%M JST}`"
    header_md += f"""

Default label: `{event_row.default_threshold_label}`

{REVIEW_THRESHOLD_PROFILE} label: `{event_row.selected_profile_event_label}`"""
    display(Markdown(header_md))

    quicklook_path = first_existing_path(QUICKLOOK_DIR, quicklook_filename(event_index, event_peak))
    olr_path = first_existing_path(OLR_DIR, quicklook_filename(event_index, event_peak))
    satellite_path = first_existing_path(SATELLITE_DIR, satellite_filename(event_index, event_peak))

    found_any_panel = False
    for title, panel_path in [
        ("Convergence quicklook", quicklook_path),
        ("OLR / cloud-proxy panel", olr_path),
        ("MODIS daily satellite panel", satellite_path),
    ]:
        if panel_path is None:
            print(f"{title}: not found locally or in Drive cache for event {event_index:03d}.")
            continue
        found_any_panel = True
        display(Markdown(f"""**{title}**

`{panel_path}`"""))
        display(Image(filename=str(panel_path), width=700))

    if not found_any_panel:
        print(f"No quicklook / OLR / satellite panels were found for event {event_index:03d}.")


In [ ]:
required_globals = [
    "selected_group_id",
    "selected_group_row",
    "representative_time_specs",
    "REVIEW_PLOT_DIR",
    "SAVE_PLOTS",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 selection cells first. Missing globals: {missing_globals}")

if not representative_time_specs:
    print("No representative synoptic times are available for the selected timing group.")
else:
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

    for time_role, analysis_time in representative_time_specs:
        snapshot_300 = load_snapshot(
            era5_runtime_ds,
            analysis_time,
            variables=["u_component_of_wind", "v_component_of_wind", "geopotential"],
            domain=UPPER_CONTEXT_DOMAIN,
            level=300,
        )
        snapshot_500 = load_snapshot(
            era5_runtime_ds,
            analysis_time,
            variables=["geopotential"],
            domain=UPPER_CONTEXT_DOMAIN,
            level=500,
        )
        snapshot_850 = load_snapshot(
            era5_runtime_ds,
            analysis_time,
            variables=["temperature"],
            domain=PRESSURE_CONTEXT_DOMAIN,
            level=850,
        )
        surface_snapshot = load_snapshot(
            era5_runtime_ds,
            analysis_time,
            variables=["mean_sea_level_pressure"],
            domain=PRESSURE_CONTEXT_DOMAIN,
        )

        wind_speed_300 = compute_wind_speed_field(snapshot_300)
        z300 = compute_geopotential_height_field(snapshot_300)
        z500 = compute_geopotential_height_field(snapshot_500)
        temp_grad_850 = compute_temperature_gradient_magnitude(snapshot_850) * 1e5
        temp_grad_850 = temp_grad_850.rename("temperature_gradient_display")
        msl_hpa = (surface_snapshot["mean_sea_level_pressure"] / 100.0).rename("msl_hpa")

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(18.2, 6.1),
            subplot_kw={"projection": ccrs.PlateCarree()},
        )
        fig.subplots_adjust(top=0.84, bottom=0.18, left=0.04, right=0.99, wspace=0.08)

        # Panel 1: 300 hPa isotachs + height contours.
        ax = axes[0]
        configure_map_axis(ax, UPPER_CONTEXT_DOMAIN)
        wind_levels = np.arange(20.0, 85.0, 5.0)
        wind_mesh = ax.contourf(
            wind_speed_300.longitude,
            wind_speed_300.latitude,
            wind_speed_300,
            levels=wind_levels,
            cmap="viridis",
            extend="max",
            transform=ccrs.PlateCarree(),
        )
        z300_contours = ax.contour(
            z300.longitude,
            z300.latitude,
            z300,
            levels=rounded_contour_levels(z300, step=120.0),
            colors="black",
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
        )
        ax.clabel(z300_contours, inline=True, fontsize=7, fmt="%d")
        add_region_outlines(ax)
        ax.set_title("300 hPa isotachs with geopotential-height contours", fontsize=11)

        # Panel 2: 500 hPa geopotential height contours.
        ax = axes[1]
        configure_map_axis(ax, UPPER_CONTEXT_DOMAIN)
        z500_contours = ax.contour(
            z500.longitude,
            z500.latitude,
            z500,
            levels=rounded_contour_levels(z500, step=60.0),
            colors="#303030",
            linewidths=0.9,
            transform=ccrs.PlateCarree(),
        )
        ax.clabel(z500_contours, inline=True, fontsize=7, fmt="%d")
        add_region_outlines(ax)
        ax.set_title("500 hPa geopotential height contours [gpm]", fontsize=11)

        # Panel 3: surface pressure + temperature-gradient shading.
        ax = axes[2]
        configure_map_axis(ax, PRESSURE_CONTEXT_DOMAIN)
        tempgrad_levels = np.arange(0.0, 10.5, 0.5)
        tempgrad_mesh = ax.contourf(
            temp_grad_850.longitude,
            temp_grad_850.latitude,
            temp_grad_850,
            levels=tempgrad_levels,
            cmap="magma",
            extend="max",
            transform=ccrs.PlateCarree(),
        )
        msl_contours = ax.contour(
            msl_hpa.longitude,
            msl_hpa.latitude,
            msl_hpa,
            levels=np.arange(960.0, 1048.0, 4.0),
            colors="black",
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
        )
        ax.clabel(msl_contours, inline=True, fontsize=7, fmt="%d")
        add_region_outlines(ax)
        ax.set_title("Surface MSLP with 850 hPa temperature-gradient shading", fontsize=11)

        # Horizontal colorbars beneath shaded panels.
        for target_ax, mappable, label in [
            (axes[0], wind_mesh, "300 hPa wind speed [m s^-1] (display-only cap)"),
            (axes[2], tempgrad_mesh, "850 hPa |grad T| [K (100 km)^-1] (display-only cap)"),
        ]:
            bbox = target_ax.get_position()
            cbar_ax = fig.add_axes([bbox.x0, 0.08, bbox.width, 0.025])
            cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal")
            cbar.set_label(label)

        time_jst = pd.Timestamp(analysis_time) + pd.Timedelta(hours=9)
        fig.suptitle(
            f"Timing group {selected_group_id} | forecast-funnel context | {time_role} | {pd.Timestamp(analysis_time):%Y-%m-%d %H:%M UTC} / {time_jst:%Y-%m-%d %H:%M JST}",
            fontsize=13,
            y=0.98,
        )

        if SAVE_PLOTS:
            safe_role = time_role.replace(" ", "_").replace("/", "-")
            output_path = REVIEW_PLOT_DIR / f"{selected_group_id}_{safe_role}_synoptic_funnel.png"
            fig.savefig(output_path, dpi=180, bbox_inches="tight")
            maybe_copy_to_drive(output_path, verbose=False)
        plt.show()


In [ ]:
required_globals = [
    "selected_group_id",
    "selected_group_row",
    "selected_group_timeseries_df",
    "selected_profile_event_df",
    "selected_threshold_row",
    "REVIEW_THRESHOLD_PROFILE",
    "STATE_COLOR_MAP",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 selection cells first. Missing globals: {missing_globals}")

if selected_group_timeseries_df.empty:
    print("No labeled continuous timing-group time series were found for the selected group/profile.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(11.6, 7.8), sharex=True)
    fig.subplots_adjust(top=0.82, bottom=0.18, left=0.10, right=0.96, hspace=0.16)

    for _, row in selected_group_timeseries_df.iterrows():
        color = STATE_COLOR_MAP.get(str(row["continuous_regime_label"]), "#dddddd")
        for ax in axes:
            ax.axvspan(
                row["time"],
                row["time"] + pd.Timedelta(hours=CONTINUOUS_TIME_STEP_HOURS),
                color=color,
                alpha=0.14,
                zorder=0,
            )

    # Event-peak markers.
    for event_row in selected_profile_event_df.itertuples(index=False):
        for ax in axes:
            ax.axvline(pd.Timestamp(event_row.event_peak), color="black", linestyle=":", linewidth=1.0, alpha=0.75)

    offshore_time = selected_group_row.get("first_offshore_support_time", pd.NaT)
    coastal_time = selected_group_row.get("first_coastal_support_time", pd.NaT)
    if pd.notna(offshore_time):
        for ax in axes:
            ax.axvline(pd.Timestamp(offshore_time), color="#4c78a8", linestyle="--", linewidth=1.3, alpha=0.9)
    if pd.notna(coastal_time):
        for ax in axes:
            ax.axvline(pd.Timestamp(coastal_time), color="#f58518", linestyle="--", linewidth=1.3, alpha=0.9)

    axes[0].plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["polygon_qflux_850_mean"],
        color="#2f4b7c",
        linewidth=2.0,
        label="Polygon moist-ascent proxy",
    )
    axes[0].plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["coastal_qflux_850_mean"],
        color="#f28e2b",
        linewidth=2.0,
        label="Coastal-only moist-ascent proxy",
    )
    axes[0].axhline(float(selected_threshold_row["polygon_qflux_min"]), color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    axes[0].axhline(float(selected_threshold_row["coastal_qflux_split"]), color="#f28e2b", linestyle="--", linewidth=1.2, alpha=0.8)
    axes[0].set_ylabel("850 hPa q × (-omega) [1e-3 Pa s^-1]")
    axes[0].grid(alpha=0.25)

    axes[1].plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["polygon_div_925_mean"],
        color="#2f4b7c",
        linewidth=2.0,
        label="Polygon divergence",
    )
    axes[1].plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["coastal_div_925_mean"],
        color="#f28e2b",
        linewidth=2.0,
        label="Coastal-only divergence",
    )
    axes[1].axhline(float(selected_threshold_row["polygon_div_max"]), color="#2f4b7c", linestyle="--", linewidth=1.2, alpha=0.8)
    axes[1].axhline(float(selected_threshold_row["coastal_div_max"]), color="#f28e2b", linestyle="--", linewidth=1.2, alpha=0.8)
    axes[1].set_ylabel("925 hPa divergence [1e-5 s^-1]")
    axes[1].set_xlabel("Time")
    axes[1].grid(alpha=0.25)

    line_handles = [
        plt.Line2D([0], [0], color="#2f4b7c", linewidth=2.0, label="Polygon series"),
        plt.Line2D([0], [0], color="#f28e2b", linewidth=2.0, label="Coastal-only series"),
        plt.Line2D([0], [0], color="black", linestyle=":", linewidth=1.0, label="Event peak"),
        plt.Line2D([0], [0], color="#4c78a8", linestyle="--", linewidth=1.3, label="First offshore-support onset"),
        plt.Line2D([0], [0], color="#f58518", linestyle="--", linewidth=1.3, label="First coastal-support onset"),
    ]
    fig.legend(line_handles, [handle.get_label() for handle in line_handles], loc="upper center", bbox_to_anchor=(0.5, 0.92), ncol=3, frameon=False)

    state_handles = [
        Patch(facecolor=color, alpha=0.14, label=label)
        for label, color in STATE_COLOR_MAP.items()
    ]
    fig.legend(state_handles, [handle.get_label() for handle in state_handles], loc="lower center", bbox_to_anchor=(0.5, 0.03), ncol=4, frameon=False)

    fig.suptitle(
        f"Timing group {selected_group_id} | continuous evolution | {REVIEW_THRESHOLD_PROFILE} | {selected_group_row['continuous_spell_regime_path']}",
        fontsize=13,
        y=0.98,
    )
    plt.show()


In [ ]:
required_globals = [
    "catalog_spell_events_df",
    "continuous_summary_df",
    "TIMING_GROUP_REVIEW_SCAFFOLD_PATH",
    "selected_group_id",
    "REVIEW_THRESHOLD_PROFILE",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 cells first. Missing globals: {missing_globals}")

event_sequence_summary_df = (
    catalog_spell_events_df
    .sort_values(["catalog_spell_id", "event_peak"])
    .groupby("catalog_spell_id", sort=False)
    .agg(
        grouped_event_count=("event_index", "size"),
        event_indices=("event_index", lambda values: ", ".join(str(int(value)) for value in values)),
        event_level_sequence=("objective_regime_label", lambda values: " -> ".join(str(value) for value in values)),
    )
    .reset_index()
)

timing_group_review_scaffold_df = continuous_summary_df.merge(
    event_sequence_summary_df,
    on="catalog_spell_id",
    how="left",
)
manual_defaults = {
    "manual_transition_verified": "",
    "manual_transition_type": "",
    "manual_transition_confidence": "",
    "manual_offshore_onset_jst": "",
    "manual_coastal_onset_jst": "",
    "manual_transition_lag_hours": "",
    "satellite_supports_transition": "",
    "forecast_funnel_supports_transition": "",
    "cross_section_priority": "",
    "cross_section_start_lon": "",
    "cross_section_start_lat": "",
    "cross_section_end_lon": "",
    "cross_section_end_lat": "",
    "coastal_impact_visible": "",
    "coastal_impact_region": "",
    "manual_notes": "",
}
for column_name, default_value in manual_defaults.items():
    if column_name not in timing_group_review_scaffold_df.columns:
        timing_group_review_scaffold_df[column_name] = default_value

timing_group_review_scaffold_df = timing_group_review_scaffold_df.sort_values(
    ["threshold_profile", "catalog_spell_id"]
).reset_index(drop=True)

timing_group_review_scaffold_df.to_csv(TIMING_GROUP_REVIEW_SCAFFOLD_PATH, index=False)
maybe_copy_to_drive(TIMING_GROUP_REVIEW_SCAFFOLD_PATH)

selected_scaffold_row_df = timing_group_review_scaffold_df.loc[
    (timing_group_review_scaffold_df["catalog_spell_id"].astype(str) == selected_group_id)
    & (timing_group_review_scaffold_df["threshold_profile"] == REVIEW_THRESHOLD_PROFILE)
]

print("Timing-group manual review scaffold saved")
print(f"- {TIMING_GROUP_REVIEW_SCAFFOLD_PATH}")
print("Selected timing-group scaffold row")
display(selected_scaffold_row_df[[
    "catalog_spell_id",
    "threshold_profile",
    "continuous_spell_regime_path",
    "continuous_clear_sequence",
    "grouped_event_count",
    "event_indices",
    "event_level_sequence",
    "manual_transition_verified",
    "manual_transition_type",
    "manual_transition_confidence",
    "satellite_supports_transition",
    "forecast_funnel_supports_transition",
    "cross_section_priority",
    "coastal_impact_visible",
    "coastal_impact_region",
    "manual_notes",
]])


# Cross-Section Layer (Scaffold For Next Pass)

The cross-section layer is intentionally reserved for the next implementation pass so the first Notebook 26 version stays stable and readable.

## Planned cross-section families

### 1. Front-normal / jet-normal pressure section

Purpose:
- show potential-temperature bunching / baroclinicity
- show sloping ascent and descent around the jet
- compare isotachs with vertical-motion structure

Planned plot elements:
- x-axis: distance along section `[km]`
- y-axis: pressure `[hPa]`
- black contours: potential temperature, `theta [K]`
- shading: vertical motion, `omega [Pa s^-1]` or a closely related ascent diagnostic
- red contours: wind speed `[m s^-1]`
- optional vectors: along-section flow plus scaled vertical motion
- terrain or below-ground mask at the bottom when available

### 2. Potential-vorticity / tropopause pressure section

Purpose:
- show high-PV intrusion / tropopause folding near the jet
- compare PV structure with the ascent / descent couplet

Planned plot elements:
- shading: potential vorticity, `PV [PVU]`
- thick contour: `2 PVU` dynamic tropopause
- black contours: potential temperature, `theta [K]`
- optional red contours: isotachs `[m s^-1]`
- optional omega contours or circulation vectors

### 3. Isentropic diagnostic

For the first implementation pass, the preferred isentropic companion product is a **plan-view isentropic-surface map** on selected `theta` levels rather than a full theta-coordinate vertical section.

Planned products:
- pressure on a selected isentropic surface
- wind on that isentropic surface
- optional PV or moisture on that isentropic surface

## Endpoint capture

The manual-review scaffold now includes fields for cross-section endpoints:
- `cross_section_start_lon`
- `cross_section_start_lat`
- `cross_section_end_lon`
- `cross_section_end_lat`

The intended first use is typed endpoints, not click-to-select endpoints.


In [ ]:
required_globals = [
    "selected_group_id",
    "REVIEW_THRESHOLD_PROFILE",
    "TIMING_GROUP_REVIEW_SCAFFOLD_PATH",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 cells first. Missing globals: {missing_globals}")

print("Notebook 26 summary")
print(f"- Selected timing group: {selected_group_id}")
print(f"- Review threshold profile: {REVIEW_THRESHOLD_PROFILE}")
print("- First-look event maps, saved imagery panels, synoptic forecast-funnel panels, and continuous timing-group diagnostics are active in this first pass.")
print("- Cross sections are intentionally scaffolded for the next implementation layer rather than half-implemented here.")
print(f"- Manual review scaffold: {TIMING_GROUP_REVIEW_SCAFFOLD_PATH}")
print()
print("Suggested review order for this timing group")
print("1. Check the event membership table and threshold passes.")
print("2. Review the event-peak divergence and moist-ascent proxy maps for each event.")
print("3. Check the saved quicklook / OLR / satellite panels for cloud-band structure.")
print("4. Review the broader forecast-funnel panels to place the timing group in synoptic context.")
print("5. Compare all of that against the continuous timing-group evolution panel from Notebook 25.")
print("6. Fill in the scaffold row with manual transition and impact notes.")
